# T5 · DimRoute, DimMode, DimStatus, DimDate
Subtasks: Block A (trap + DimRoute), Block B (DimMode, DimStatus), Block C (DimDate), four questions. All in this one notebook.

In [1]:
import pandas as pd, numpy as np, os
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_rows", 100)

p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p, parse_dates=["ShipDate"])

# DimCity from T4 (alias -> CityKey) so this notebook runs on its own. Karachi = 1.
city_key = {"Karachi":1,"Gwadar":2,"Lahore":3,"Multan":4,"Islamabad":5,"Quetta":6,"Faisalabad":7,
            "Rawalpindi":8,"Sialkot":9,"Peshawar":10,"Hyderabad":11,"Sukkur":12,"Gujranwala":13,"Abbottabad":14}
alias_city = {"Karachi Port":"Karachi","Port Qasim":"Karachi","Karachi Cantt":"Karachi","Pipri Marshalling Yard":"Karachi","Karachi":"Karachi",
              "Gwadar Port":"Gwadar","Lahore":"Lahore","Lahore Dry Port":"Lahore","Multan":"Multan","Islamabad":"Islamabad",
              "Islamabad Dry Port":"Islamabad","Quetta":"Quetta","Faisalabad":"Faisalabad","Faisalabad Dry Port":"Faisalabad",
              "Rawalpindi":"Rawalpindi","Sialkot Dry Port":"Sialkot","Sialkot":"Sialkot","Peshawar":"Peshawar","Peshawar Cantt":"Peshawar",
              "Hyderabad":"Hyderabad","Sukkur":"Sukkur","Gujranwala":"Gujranwala","Abbottabad":"Abbottabad"}
province = {"Karachi":"Sindh","Hyderabad":"Sindh","Sukkur":"Sindh","Gwadar":"Balochistan","Quetta":"Balochistan",
            "Lahore":"Punjab","Faisalabad":"Punjab","Multan":"Punjab","Sialkot":"Punjab","Gujranwala":"Punjab","Rawalpindi":"Punjab",
            "Islamabad":"Islamabad Capital Territory","Peshawar":"Khyber Pakhtunkhwa","Abbottabad":"Khyber Pakhtunkhwa"}
alias_key = {a: city_key[c] for a, c in alias_city.items()}
key_city  = {v: k for k, v in city_key.items()}

## Block A · The trap, then DimRoute

In [2]:
raw_pairs = src[["Origin", "Destination"]].drop_duplicates()
print("Remove Duplicates on the raw strings            :", len(raw_pairs), "rows")

f = src.copy()
f["OriginCityKey"] = f.Origin.map(alias_key)
f["DestCityKey"]   = f.Destination.map(alias_key)
conformed = f[["OriginCityKey", "DestCityKey"]].drop_duplicates()
print("Conform to CityKeys first, then Remove Duplicates:", len(conformed), "rows")
assert len(raw_pairs) == 30 and len(conformed) == 20

# which raw spellings collapsed into the same route
g = f.groupby(["OriginCityKey", "DestCityKey"]).apply(
        lambda d: sorted(set(d.Origin + "  ->  " + d.Destination)), include_groups=False)
g = g[g.map(len) > 1]
g.rename("raw pairs that are the same route").reset_index().assign(
    Route=lambda d: d.OriginCityKey.map(key_city) + " -> " + d.DestCityKey.map(key_city))[["Route", "raw pairs that are the same route"]]

Remove Duplicates on the raw strings            : 30 rows
Conform to CityKeys first, then Remove Duplicates: 20 rows


,Route,raw pairs that are the same route
0,Karachi -> Lahore,"[Karachi Cantt -> Lahore, Karachi Port -> Lahore Dry Port, Pipri Marshalling Yard -> Lahore, Port Qasim -> Lahore Dry Port]"
1,Karachi -> Multan,"[Karachi Cantt -> Multan, Pipri Marshalling Yard -> Multan, Port Qasim -> Multan]"
2,Karachi -> Faisalabad,"[Pipri Marshalling Yard -> Faisalabad, Port Qasim -> Faisalabad Dry Port]"
3,Karachi -> Rawalpindi,"[Karachi Cantt -> Rawalpindi, Pipri Marshalling Yard -> Rawalpindi]"
4,Karachi -> Peshawar,"[Karachi Cantt -> Peshawar Cantt, Karachi Port -> Peshawar]"
5,Karachi -> Hyderabad,"[Karachi -> Hyderabad, Karachi Port -> Hyderabad]"
6,Karachi -> Sukkur,"[Karachi -> Sukkur, Karachi Cantt -> Sukkur]"


**The trap:** Remove Duplicates on the raw text only collapses identical spellings, and the three source systems spell the same city differently (`Karachi Port`, `Karachi Cantt`, `Pipri Marshalling Yard` are all Karachi), so 30 raw pairs survive. **Order of operations:** conform first (map each raw string to its `CityKey` using the T4 map), then deduplicate on `(OriginCityKey, DestCityKey)` to get 20. Dedupe-then-conform would leave 30 route rows that mean 20 routes, and duplicate route names in the dimension.

In [3]:
# DistanceBand: straight-line distance between approximate city coordinates (editorial, not road distance)
coord = {"Karachi":(24.86,67.01),"Gwadar":(25.12,62.33),"Lahore":(31.55,74.34),"Multan":(30.20,71.47),"Islamabad":(33.68,73.05),
         "Quetta":(30.18,66.99),"Faisalabad":(31.42,73.08),"Rawalpindi":(33.60,73.05),"Sialkot":(32.49,74.53),"Peshawar":(34.01,71.58),
         "Hyderabad":(25.40,68.37),"Sukkur":(27.70,68.86),"Gujranwala":(32.16,74.19),"Abbottabad":(34.15,73.21)}
def km(a, b):
    la1, lo1, la2, lo2 = map(np.radians, (*coord[a], *coord[b]))
    h = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(h))
band = lambda d: "Short (<300 km)" if d < 300 else "Medium (300-800 km)" if d <= 800 else "Long (>800 km)"

routes = conformed.reset_index(drop=True)          # first-appearance order in the data
routes.insert(0, "RouteKey", range(1, len(routes) + 1))
routes["Orig"] = routes.OriginCityKey.map(key_city)
routes["Dest"] = routes.DestCityKey.map(key_city)
assert (routes.OriginCityKey != routes.DestCityKey).all()
routes["RouteName"]    = routes.Orig + " \u2192 " + routes.Dest
routes["Corridor"]     = routes.Orig.map(province) + " \u2192 " + routes.Dest.map(province)
routes["DistanceBand"] = [band(km(o, d)) for o, d in zip(routes.Orig, routes.Dest)]

unknown = pd.DataFrame([{"RouteKey": -1, "OriginCityKey": -1, "DestCityKey": -1, "RouteName": "Unknown", "Corridor": "Unknown", "DistanceBand": "Unknown"}])
DimRoute = pd.concat([routes[["RouteKey", "OriginCityKey", "DestCityKey", "RouteName", "Corridor", "DistanceBand"]], unknown], ignore_index=True)
assert DimRoute.RouteName.nunique() == 21 and (DimRoute.RouteKey == -1).sum() == 1
DimRoute

,RouteKey,OriginCityKey,DestCityKey,RouteName,Corridor,DistanceBand
0,1,1,3,Karachi → Lahore,Sindh → Punjab,Long (>800 km)
1,2,1,7,Karachi → Faisalabad,Sindh → Punjab,Long (>800 km)
2,3,2,6,Gwadar → Quetta,Balochistan → Balochistan,Medium (300-800 km)
3,4,1,4,Karachi → Multan,Sindh → Punjab,Medium (300-800 km)
4,5,1,9,Karachi → Sialkot,Sindh → Punjab,Long (>800 km)
5,6,1,10,Karachi → Peshawar,Sindh → Khyber Pakhtunkhwa,Long (>800 km)
6,7,2,1,Gwadar → Karachi,Balochistan → Sindh,Medium (300-800 km)
7,8,1,5,Karachi → Islamabad,Sindh → Islamabad Capital Territory,Long (>800 km)
8,9,1,11,Karachi → Hyderabad,Sindh → Sindh,Short (<300 km)
9,10,1,8,Karachi → Rawalpindi,Sindh → Punjab,Long (>800 km)


## Block B · DimMode and DimStatus (reference data, typed by hand)

In [4]:
leg = src.groupby("SourceSystem").agg(legs=("ShipmentID", "count"), avg_kg=("WeightKG", "mean"), avg_cost=("CostPKR", "mean")).round(0)
print(leg)      # evidence behind TonnageClass and CostProfile

DimMode = pd.DataFrame([
 (1,  "Marine",  "Marine carrier", "Medium (about 17 t per leg)", "Medium cost per leg",  "Y"),
 (2,  "Rail",    "Rail carrier",   "Heavy (about 29 t per leg)",  "High cost per leg",    "N"),
 (3,  "Road",    "Road carrier",   "Light (about 6 t per leg)",   "Low cost per leg",     "N"),
 (-1, "Unknown", "Unknown",        "Unknown",                     "Unknown",              "N"),
], columns=["ModeKey", "ModeName", "Carrier", "TonnageClass", "CostProfile", "Containerised?"])

DimStatus = pd.DataFrame([
 (1,  "Delivered",  True,  False, 3),
 (2,  "In Transit", False, False, 1),
 (3,  "Delayed",    False, True,  2),
 (-1, "Unknown",    False, False, 99),
], columns=["StatusKey", "StatusName", "IsClosed", "IsException", "DisplayOrder"])

assert set(src.SourceSystem) <= set(DimMode.ModeName) and set(src.Status) <= set(DimStatus.StatusName)
assert (DimMode.ModeKey == -1).sum() == 1 and (DimStatus.StatusKey == -1).sum() == 1
display(DimMode); display(DimStatus)

              legs   avg_kg  avg_cost
SourceSystem                         
Marine          12  17208.0  134833.0
Rail            10  29525.0  258500.0
Road            12   6408.0   50417.0


,ModeKey,ModeName,Carrier,TonnageClass,CostProfile,Containerised?
0,1,Marine,Marine carrier,Medium (about 17 t per leg),Medium cost per leg,Y
1,2,Rail,Rail carrier,Heavy (about 29 t per leg),High cost per leg,N
2,3,Road,Road carrier,Light (about 6 t per leg),Low cost per leg,N
3,-1,Unknown,Unknown,Unknown,Unknown,N


,StatusKey,StatusName,IsClosed,IsException,DisplayOrder
0,1,Delivered,True,False,3
1,2,In Transit,False,False,1
2,3,Delayed,False,True,2
3,-1,Unknown,False,False,99


## Block C · DimDate (all 28 days of February 2026)

In [5]:
days = pd.date_range("2026-02-01", "2026-02-28", freq="D")
holidays = {pd.Timestamp("2026-02-05")}            # Kashmir Solidarity Day (public holiday); edit if your calendar differs
per_day = src.groupby("ShipDate").size()

DimDate = pd.DataFrame({
    "DateKey":       days.strftime("%Y%m%d").astype(int),
    "FullDate":      days,
    "Year":          days.year,
    "Quarter":       days.quarter,
    "MonthNumber":   days.month,
    "MonthName":     days.strftime("%B"),
    "DayOfMonth":    days.day,
    "DayOfWeekNumber": days.dayofweek + 1,         # Monday = 1
    "DayName":       days.strftime("%A"),
    "IsWeekend":     days.dayofweek >= 5,
    "IsHoliday":     days.isin(holidays),
})
DimDate["Shipments that day"] = DimDate.FullDate.map(per_day).fillna(0).astype(int)
DimDate["In fact table?"]     = np.where(DimDate["Shipments that day"] > 0, "Yes", "No")

rows_dd  = len(DimDate); busy = int((DimDate["In fact table?"] == "Yes").sum()); quiet = rows_dd - busy
assert (rows_dd, busy, quiet) == (28, 12, 16) and DimDate.DateKey.is_unique
print("Rows in DimDate:", rows_dd, "| days with shipments:", busy, "| quiet days:", quiet)
print("Average daily tonnage, 12 shipping days:", round(src.WeightKG.sum() / busy), "kg")
print("Average daily tonnage, 28 calendar days:", round(src.WeightKG.sum() / rows_dd), "kg")
DimDate

Rows in DimDate: 28 | days with shipments: 12 | quiet days: 16
Average daily tonnage, 12 shipping days: 48221 kg
Average daily tonnage, 28 calendar days: 20666 kg


,DateKey,FullDate,Year,Quarter,MonthNumber,MonthName,DayOfMonth,DayOfWeekNumber,DayName,IsWeekend,IsHoliday,Shipments that day,In fact table?
0,20260201,2026-02-01,2026,1,2,February,1,7,Sunday,True,False,1,Yes
1,20260202,2026-02-02,2026,1,2,February,2,1,Monday,False,False,3,Yes
2,20260203,2026-02-03,2026,1,2,February,3,2,Tuesday,False,False,4,Yes
3,20260204,2026-02-04,2026,1,2,February,4,3,Wednesday,False,False,3,Yes
4,20260205,2026-02-05,2026,1,2,February,5,4,Thursday,False,True,3,Yes
5,20260206,2026-02-06,2026,1,2,February,6,5,Friday,False,False,3,Yes
6,20260207,2026-02-07,2026,1,2,February,7,6,Saturday,True,False,3,Yes
7,20260208,2026-02-08,2026,1,2,February,8,7,Sunday,True,False,0,No
8,20260209,2026-02-09,2026,1,2,February,9,1,Monday,False,False,3,Yes
9,20260210,2026-02-10,2026,1,2,February,10,2,Tuesday,False,False,3,Yes


## Lookups: every fact row must find its keys (no -1)

In [6]:
route_key = routes.set_index(["OriginCityKey", "DestCityKey"]).RouteKey
f["RouteKey"]  = [route_key.get((o, d), -1) for o, d in zip(f.OriginCityKey, f.DestCityKey)]
f["ModeKey"]   = f.SourceSystem.map(DimMode.set_index("ModeName").ModeKey).fillna(-1).astype(int)
f["StatusKey"] = f.Status.map(DimStatus.set_index("StatusName").StatusKey).fillna(-1).astype(int)
f["DateKey"]   = f.ShipDate.dt.strftime("%Y%m%d").astype(int)
f["DateKey"]   = f.DateKey.where(f.DateKey.isin(DimDate.DateKey), -1)

res = pd.DataFrame([
 ("Rows in DimRoute (excluding Unknown)", 20, len(routes)),
 ("Distinct RouteName values",            21, DimRoute.RouteName.nunique()),
 ("Rows in DimDate",                      28, len(DimDate)),
 ("Days with at least one shipment",      12, busy),
 ("Quiet days",                           16, quiet),
 ("Fact rows with any key = -1",           0, int((f[["RouteKey", "ModeKey", "StatusKey", "DateKey"]] == -1).any(axis=1).sum())),
 ("Total WeightKG",                  578_650, int(f.WeightKG.sum())),
], columns=["Check", "Expected", "Result"])
res["Pass?"] = (res.Expected == res.Result).map({True: "pass", False: "FIX"})
assert (res["Pass?"] == "pass").all()
res

,Check,Expected,Result,Pass?
0,Rows in DimRoute (excluding Unknown),20,20,pass
1,Distinct RouteName values,21,21,pass
2,Rows in DimDate,28,28,pass
3,Days with at least one shipment,12,12,pass
4,Quiet days,16,16,pass
5,Fact rows with any key = -1,0,0,pass
6,Total WeightKG,578650,578650,pass


## The four questions

**Q1. DimRoute holds two city keys. Does that make DimCity a snowflake?**

No: `FactShipment` joins to `DimCity` directly (twice, as origin and as destination), so `DimCity` is not a second layer hanging off another dimension. The two city keys in `DimRoute` are only descriptive links that define what a route is; it would become a snowflake only if reports could reach `DimCity` solely by going through `DimRoute`.

**Q2. Karachi → Lahore is served by both marine and rail. Should Mode be a column in DimRoute or its own dimension?**

In [7]:
multi = f.groupby("RouteKey").ModeKey.nunique()
print("Routes served by more than one mode:", int((multi > 1).sum()), "of", len(routes))
print(f[f.RouteKey == 1].groupby("SourceSystem").size().to_dict(), "<- legs on", routes.loc[0, "RouteName"])

Routes served by more than one mode: 6 of 20
{'Marine': 4, 'Rail': 3} <- legs on Karachi → Lahore


Its own dimension. A route is a pair of cities; the mode is how a particular leg travelled. Putting Mode inside `DimRoute` splits Karachi → Lahore into one row per mode, so `RouteName` stops being unique, every new mode multiplies the route rows, and the two mode-specific questions (route totals, mode totals) both get harder. Mode also carries its own attributes (carrier, tonnage class, cost profile), which belong in `DimMode`, stored once.

**Q3. A Karachi → Lahore service routed via Sukkur: same two cities, different path. New DimRoute row?** *(arguable)*
My position: **yes, a new row**, but keep the city pair queryable. Add `ViaCity` (existing rows get "Direct") and an `ODPairName` attribute ("Karachi → Lahore") on every row. Then route-level reports separate the two paths when distance, cost or transit time differ (they are different products with different economics), while city-pair reports still roll both up. The opposing argument: the business defines a route by its two cities, and a path is a leg-level detail, so a new row fragments "Karachi → Lahore" totals and makes every route report carry a rollup. That is a fair objection if nobody will ever compare the paths; if anyone will, the extra row is cheaper than losing the information, because history cannot be re-cut by a path that was never recorded.

**Q4. Why does DimDate get 28 rows when the fact table only touches 12?**
A date dimension describes the calendar, not the transactions. With only the 12 shipping dates, "average daily tonnage in February" computes 578,650 / 12 = 48,221 kg; against the real calendar it is 578,650 / 28 = 20,666 kg, and only the second answers the CFO. Quiet days must also exist so reports can show zero-shipment days, and so time-intelligence functions (rolling windows, month-to-date, same period last month) work on a contiguous calendar. In production the dimension covers whole years (built once), so it also serves other facts (fuel purchases, complaints, stock snapshots) and dates not yet loaded.